<a href="https://colab.research.google.com/github/SVz54/9517/blob/draft1/9517.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# --- Kaggle download (same as before) ---
import os, json, shutil, zipfile, glob, pathlib

os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json') and os.path.exists('/content/kaggle.json'):
    shutil.move('/content/kaggle.json', '/root/.kaggle/kaggle.json')
!chmod 600 /root/.kaggle/kaggle.json

!pip -q install kaggle
DATA_DIR = "/content/AgroPest12"
os.makedirs(DATA_DIR, exist_ok=True)
!kaggle datasets download -d rupankarmajumdar/crop-pests-dataset -p $DATA_DIR -q

# Unzip (overwrite if re-running)
zip_files = glob.glob(f"{DATA_DIR}/*.zip")
assert zip_files, "Zip not found – did the Kaggle download succeed?"
zip_path = zip_files[0]
!unzip -q -o "$zip_path" -d "$DATA_DIR"

# --- Auto-detect BASE: the folder that contains train/valid/test with images+labels ---
def find_yolo_base(root):
    for p, d, f in os.walk(root):
        if (os.path.isdir(os.path.join(p, "train", "images")) and
            os.path.isdir(os.path.join(p, "train", "labels")) and
            os.path.isdir(os.path.join(p, "valid", "images")) and
            os.path.isdir(os.path.join(p, "valid", "labels"))):
            return p
    return None

BASE = find_yolo_base(DATA_DIR)
assert BASE is not None, f"Could not find YOLO folders under {DATA_DIR}. Found: {os.listdir(DATA_DIR)}"
print("BASE:", BASE)
print("train samples:", len(glob.glob(os.path.join(BASE, "train/images/*.jpg"))))
print("val samples:", len(glob.glob(os.path.join(BASE, "valid/images/*.jpg"))))
print("test samples:", len(glob.glob(os.path.join(BASE, "test/images/*.jpg"))))
BASE = pathlib.Path(BASE)  # keep as Path for later cells

Dataset URL: https://www.kaggle.com/datasets/rupankarmajumdar/crop-pests-dataset
License(s): MIT
BASE: /content/AgroPest12
train samples: 11502
val samples: 1095
test samples: 546


In [ ]:
!pip -q install --upgrade ultralytics==8.3.20 opencv-python-headless==4.10.0.84 matplotlib==3.9.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 125.1 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
yaml_path = BASE / "data.yaml"
txt = f"""# AgroPest-12
path: {BASE.as_posix()}
train: train/images
val: valid/images
test: test/images
names:
  0: aphid
  1: armyworm
  2: beetle
  3: bollworm
  4: grasshopper
  5: leafhopper
  6: locust
  7: mealybug
  8: mosquito
  9: moth
  10: sawfly
  11: weevil
"""
yaml_path.write_text(txt)
print("Wrote:", yaml_path)


Wrote: /content/AgroPest12/data.yaml


In [ ]:
from ultralytics import YOLO
import os, glob

model = YOLO('yolov8n.pt')   # tiny + fast; swap to yolov8s.pt later if you want

# use a handful of val images for a demo run
val_imgs = sorted(glob.glob(str(BASE / 'valid/images/*.jpg')))[:12]
print("Demo images:", len(val_imgs))

pred_root = "/content/preds"
res = model.predict(val_imgs, conf=0.25, save=True, project=pred_root, name="yolo_preds", exist_ok=True, imgsz=640)
print("Saved predicted images to:", pred_root + "/yolo_preds")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 6.25M/6.25M [00:00<00:00, 89.1MB/s]


Demo images: 12

0: 640x640 (no detections), 308.0ms
1: 640x640 (no detections), 308.0ms
2: 640x640 1 bear, 308.0ms
3: 640x640 (no detections), 308.0ms
4: 640x640 1 cat, 308.0ms
5: 640x640 1 horse, 308.0ms
6: 640x640 (no detections), 308.0ms
7: 640x640 1 person, 308.0ms
8: 640x640 1 bird, 1 horse, 308.0ms
9: 640x640 1 teddy bear, 308.0ms
10: 640x640 1 bird, 308.0ms
11: 640x640 (no detections), 308.0ms
Speed: 16.5ms preprocess, 308.0ms inference, 3.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/preds/yolo_preds
Saved predicted images to: /content/preds/yolo_preds


In [ ]:
!pip -q uninstall -y pytorch-grad-cam grad-cam || true
!pip -q install --no-cache-dir "git+https://github.com/jacobgil/pytorch-grad-cam.git"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import pytorch_grad_cam, inspect
from pytorch_grad_cam.utils import model_targets
print("grad-cam version:", getattr(pytorch_grad_cam, "__version__", "git"))
print("Has YOLOv8Target:", hasattr(model_targets, "YOLOv8Target"))


grad-cam version: git
Has YOLOv8Target: False


In [ ]:
# --- Single-image, low-memory, true Grad-CAM on YOLOv8 (crop-only, with strong normalized overlay) ---
import os, gc, cv2, torch, numpy as np, torch.nn as nn
from ultralytics import YOLO
from pytorch_grad_cam import GradCAM

# ---- Tweakable overlay params ----
LO_PCT  = 60       # lower percentile for normalization (raise to 70 if still too blue)
HI_PCT  = 99.2     # upper percentile (raise slightly for punchier reds)
HEAT_W  = 0.70     # heatmap weight in blend (0.7–0.85 looks good)
IMG_W   = 1.0 - HEAT_W

os.environ["OMP_NUM_THREADS"] = "1"

# 1) detection model (reuse your existing one if already loaded)
model_det = YOLO('yolov8n.pt')

# 2) clean model for CAM (never call .predict() on this one)
model_cam = YOLO('yolov8n.pt')
device = "cpu"                        # keep CPU for stability
model_cam.model.to(device).eval()

def last_conv2d(ultra_model):
    last = None
    for _, m in ultra_model.model.named_modules():
        if isinstance(m, nn.Conv2d):
            last = m
    return last

target_layer = last_conv2d(model_cam)
assert target_layer is not None, "No Conv2d layer found."

def load_rgb_float(path):
    bgr = cv2.imread(path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb, (rgb.astype(np.float32)/255.0)

def clamp_int(v, lo, hi):
    return max(lo, min(int(v), hi))

# Pick the FIRST val image that yields any detection with the COCO model
chosen, chosen_det = None, None
for ip in sorted((BASE/'valid/images').glob('*.jpg'))[:50]:
    r = model_det(str(ip), conf=0.35, imgsz=320)[0]
    if r.boxes is not None and len(r.boxes) > 0:
        chosen = str(ip); chosen_det = r
        break
if chosen is None:
    raise RuntimeError("No detections on first 50 val images; lower conf or try more images.")

print("Using image:", chosen)
rgb, rgbf = load_rgb_float(chosen)
h, w = rgb.shape[:2]

# Take top-confidence detection and crop around it
k = int(torch.argmax(chosen_det.boxes.conf).item())
x1, y1, x2, y2 = chosen_det.boxes.xyxy[k].tolist()
pad = 0.15
xa = clamp_int(x1 - (x2-x1)*pad, 0, w-1)
ya = clamp_int(y1 - (y2-y1)*pad, 0, h-1)
xb = clamp_int(x2 + (x2-x1)*pad, 0, w-1)
yb = clamp_int(y2 + (y2-y1)*pad, 0, h-1)

crop  = rgb[ya:yb, xa:xb]
cropf = crop.astype(np.float32)/255.0
inp   = torch.from_numpy(cropf.transpose(2,0,1)).unsqueeze(0).to(device).float()
inp.requires_grad_(True)

# Rerun detection on the crop to get a matching class index (crop frame)
r_crop = model_det(crop[..., ::-1], conf=0.25, imgsz=320)[0]  # YOLO expects BGR
if r_crop.boxes is None or len(r_crop.boxes) == 0:
    raise RuntimeError("No detection in crop; try a different image or lower conf.")
top_cls = int(r_crop.boxes.cls[torch.argmax(r_crop.boxes.conf)].item())

# Custom class-specific target: pick strongest row for that class
class YoloRowTarget:
    def __init__(self, cls_idx, model_ref):
        self.cls_idx = int(cls_idx)
        self.nc = model_ref.model.model[-1].nc
    def __call__(self, outputs):
        pred = outputs[0]  # (N, 4+nc[+obj])
        if pred.size(1) > 4 + self.nc:  # has objectness column
            obj = pred[:, 4].sigmoid()
            cls = pred[:, 5 + self.cls_idx].sigmoid()
            return (obj * cls).max()
        else:
            cls = pred[:, 4 + self.cls_idx].sigmoid()
            return cls.max()

target = YoloRowTarget(top_cls, model_cam)

# ---- Grad-CAM forward (no smoothing to save RAM) ----
cam = GradCAM(model=model_cam.model, target_layers=[target_layer])
cam_map = cam(input_tensor=inp, targets=[target], eigen_smooth=False)[0]  # HxW in [0..1], but may be flat

# ---- Strong, robust normalization + manual overlay (JET) ----
lo, hi = np.percentile(cam_map, LO_PCT), np.percentile(cam_map, HI_PCT)
cam_map = np.clip((cam_map - lo) / (hi - lo + 1e-7), 0, 1)

heat = cv2.applyColorMap((cam_map * 255).astype(np.uint8), cv2.COLORMAP_JET)
heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
vis  = np.clip(HEAT_W * heat + IMG_W * cropf, 0, 1)

# ---- Save ----
out_dir = "/content/gradcam"
os.makedirs(out_dir, exist_ok=True)
out = os.path.join(out_dir, os.path.basename(chosen).replace(".jpg","_CROP_gradcam.jpg"))
cv2.imwrite(out, cv2.cvtColor((vis * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
print("Saved:", out)

# cleanup
del inp, cam_map, crop, cropf, r_crop, chosen_det
gc.collect()



image 1/1 /content/AgroPest12/valid/images/Weevil-101-_jpg.rf.7b2714887709397bf4467aee16ea2e79.jpg: 320x320 1 bird, 68.2ms
Speed: 1.3ms preprocess, 68.2ms inference, 0.9ms postprocess per image at shape (1, 3, 320, 320)
Using image: /content/AgroPest12/valid/images/Weevil-101-_jpg.rf.7b2714887709397bf4467aee16ea2e79.jpg

0: 320x320 1 bird, 62.1ms
Speed: 5.5ms preprocess, 62.1ms inference, 1.0ms postprocess per image at shape (1, 3, 320, 320)
Saved: /content/gradcam/Weevil-101-_jpg.rf.7b2714887709397bf4467aee16ea2e79_CROP_gradcam.jpg


25